In [ ]:
!pip install -q datasets transformers accelerate bitsandbytes

In [ ]:
import os
import re
import time
import random
from decimal import Decimal, InvalidOperation

import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ID = "openai/gsm8k"
DATASET_CONFIG = "main"
DATASET_SPLIT = "test"
MAX_EXAMPLES = None
SEED = 42
BATCH_SIZE = 1
QUANTIZATION = "none"
MAX_NEW_TOKENS = 512
DISPLAY_EXAMPLES = 10

assert QUANTIZATION in ("none", "4bit"), f"Unsupported QUANTIZATION: {QUANTIZATION}"
assert isinstance(BATCH_SIZE, int) and BATCH_SIZE > 0, f"BATCH_SIZE must be a positive integer, got {BATCH_SIZE}"
assert torch.cuda.is_available(), "CUDA is not available. This notebook requires a GPU runtime (e.g. Colab T4)."

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"CUDA version: {torch.version.cuda}")
print(f"PyTorch version: {torch.__version__}")

In [ ]:
ds = load_dataset(DATASET_ID, DATASET_CONFIG, split=DATASET_SPLIT)
print(f"Full split size: {len(ds)}")

if MAX_EXAMPLES is not None:
    assert isinstance(MAX_EXAMPLES, int) and MAX_EXAMPLES > 0, f"MAX_EXAMPLES must be positive, got {MAX_EXAMPLES}"
    assert MAX_EXAMPLES <= len(ds), f"MAX_EXAMPLES={MAX_EXAMPLES} exceeds split size {len(ds)}"
    ds = ds.shuffle(seed=SEED).select(range(MAX_EXAMPLES))
    print(f"Selected {len(ds)} examples (seed={SEED})")


def extract_gold_numeric(answer_text: str) -> str:
    """Extract the numeric answer after the final #### marker in GSM8K answer."""
    match = re.search(r"####\s*(.+)$", answer_text.strip(), re.MULTILINE)
    if match:
        return match.group(1).strip()
    return ""


# Verify extraction on first example
sample_answer = ds[0]["answer"]
sample_gold = extract_gold_numeric(sample_answer)
print(f"Sample gold extraction: '{sample_answer.split(chr(10))[-1].strip()}' -> '{sample_gold}'")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

if QUANTIZATION == "4bit":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    print("Loaded model with 4-bit NF4 quantization")
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )
    print("Loaded model in FP16")

model.eval()
print(f"Model device map: {model.hf_device_map}")
print(f"Pad token id: {tokenizer.pad_token_id}")

In [ ]:
SYSTEM_PROMPT = (
    "Solve the following math problem step by step. "
    "Show your reasoning, then finish with the final numeric answer on a new line in this exact format:\n"
    "#### <numeric answer>\n"
    "Do not include any text after the #### line."
)


def build_prompt(question: str) -> str:
    """Build a chat-template prompt for a GSM8K question."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=False,
    )


def normalize_numeric(text: str) -> str:
    """Normalize a numeric string: strip whitespace, commas, currency symbols, trailing punctuation."""
    text = text.strip()
    text = re.sub(r"[\$,]", "", text)
    text = text.rstrip(". !?")
    return text.strip()


def extract_predicted_answer(generated_text: str) -> tuple[str, str]:
    """Extract predicted answer from generated text.
    Returns (predicted_answer, parse_status).
    Prefer final #### value; fall back to last numeric token.
    """
    # Try to find the final #### marker
    match = re.search(r"####\s*(.+)$", generated_text.strip(), re.MULTILINE)
    if match:
        raw = match.group(1).strip()
        normalized = normalize_numeric(raw)
        if normalized:
            return normalized, "####_match"

    # Fallback: find the last numeric token in the text
    numbers = re.findall(r"-?\d[\d,]*\.?\d*", generated_text)
    if numbers:
        normalized = normalize_numeric(numbers[-1])
        if normalized:
            return normalized, "fallback_numeric"

    return "", "parse_failure"


def answers_match(predicted: str, gold: str) -> bool:
    """Compare predicted and gold answers using decimal-safe parsing."""
    pred_norm = normalize_numeric(predicted)
    gold_norm = normalize_numeric(gold)
    if not pred_norm or not gold_norm:
        return False
    try:
        return Decimal(pred_norm) == Decimal(gold_norm)
    except (InvalidOperation, ValueError):
        return pred_norm == gold_norm


print("Helpers defined.")
# Quick sanity check
test_prompt = build_prompt("What is 2 + 3?")
print(f"Prompt length (chars): {len(test_prompt)}")

In [ ]:
results = []
total_tokens_generated = 0
num_examples = len(ds)

print(f"Evaluating {num_examples} examples in batches of {BATCH_SIZE}...")
start_time = time.time()

for batch_start in range(0, num_examples, BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, num_examples)
    batch_indices = list(range(batch_start, batch_end))
    batch_questions = [ds[i]["question"] for i in batch_indices]
    batch_answers = [ds[i]["answer"] for i in batch_indices]

    prompts = [build_prompt(q) for q in batch_questions]
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048,
    ).to(model.device)

    input_token_count = inputs["input_ids"].shape[1]

    with torch.no_grad():
        gen_outputs = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the newly generated tokens
    new_token_ids = gen_outputs[:, input_token_count:]
    generated_texts = tokenizer.batch_decode(new_token_ids, skip_special_tokens=True)

    for i, idx in enumerate(batch_indices):
        gen_text = generated_texts[i]
        gold_raw = extract_gold_numeric(batch_answers[i])
        predicted, parse_status = extract_predicted_answer(gen_text)
        correct = answers_match(predicted, gold_raw)
        num_new_tokens = new_token_ids[i].shape[0]
        total_tokens_generated += num_new_tokens

        results.append({
            "idx": idx,
            "question": batch_questions[i],
            "gold_answer": gold_raw,
            "generated_text": gen_text,
            "predicted_answer": predicted,
            "correct": correct,
            "parse_status": parse_status,
        })

    processed = batch_end
    if processed % 50 == 0 or processed == num_examples:
        elapsed = time.time() - start_time
        rate = processed / elapsed if elapsed > 0 else 0
        print(f"  [{processed}/{num_examples}] {elapsed:.1f}s ({rate:.2f} examples/s)")

elapsed = time.time() - start_time
print(f"\nEvaluation complete: {num_examples} examples in {elapsed:.1f}s")

In [ ]:
correct_count = sum(1 for r in results if r["correct"])
parse_failures = sum(1 for r in results if r["parse_status"] == "parse_failure")
accuracy = correct_count / num_examples if num_examples > 0 else 0.0
examples_per_sec = num_examples / elapsed if elapsed > 0 else 0.0

print("=" * 60)
print("AGGREGATE METRICS")
print("=" * 60)
print(f"Model:           {MODEL_ID}")
print(f"Dataset:         {DATASET_ID} ({DATASET_CONFIG}/{DATASET_SPLIT})")
print(f"Quantization:    {QUANTIZATION}")
print(f"Evaluated:       {num_examples}")
print(f"Correct:         {correct_count}")
print(f"Accuracy:        {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Parse failures:  {parse_failures}")
print(f"Elapsed:         {elapsed:.1f}s")
print(f"Throughput:      {examples_per_sec:.2f} examples/s")
print(f"Tokens generated: {total_tokens_generated}")
print("=" * 60)

print(f"\nPREVIEW (first {min(DISPLAY_EXAMPLES, num_examples)} examples):")
print("-" * 60)
for r in results[:DISPLAY_EXAMPLES]:
    status_mark = "✓" if r["correct"] else "✗"
    print(f"[{status_mark}] idx={r['idx']} | gold={r['gold_answer']} | pred={r['predicted_answer']} | parse={r['parse_status']}")
    print(f"    Q: {r['question'][:100]}{'...' if len(r['question']) > 100 else ''}")
    gen_preview = r['generated_text'].replace('\n', ' ')[:200]
    print(f"    Gen: {gen_preview}{'...' if len(r['generated_text'].replace(chr(10), ' ')) > 200 else ''}")
    print()